In [8]:
# Système, fichiers et utilitaires
import io
import json
import os
import re
import sqlite3
import sys
import unicodedata
import warnings
from pathlib import Path

# Masquage des avertissements de déprécation
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Réseau
import requests

# Manipulation et analyse de données
import numpy as np
import pandas as pd

# Données géographiques
import geopandas as gpd
from shapely.geometry import shape

# Visualisation et Data Profiling
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

# Machine Learning & Prétraitement (Scikit-Learn & XGBoost)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import BallTree
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

# Configuration d'affichage et de style
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
%matplotlib inline

print("Imports OK")

Imports OK


In [9]:
# Fonction de diagnostic global

def diagnostic_complet(df, nom_dataset="Dataset"):
  print("==================================================")
  print(f"=== DIAGNOSTIC COMPLET : {nom_dataset.upper()} ===")
  print("==================================================")

  # 1. Volumétrie et structure
  print(f"\n--- 1. VOLUMÉTRIE & STRUCTURE ---")
  print(f"Nombre de lignes : {df.shape[0]:,}")
  print(f"Nombre de colonnes : {df.shape[1]}")

  # 2. Valeurs manquantes détaillées
  print(f"\n--- 2. SYNTHÈSE DES VALEURS MANQUANTES ---")
  missing_df = pd.DataFrame(
      {
          "nuls": df.isna().sum(),
          "pourcentage": (df.isna().mean() * 100).round(2),
      }
  ).sort_values(by="nuls", ascending=False)
  display(missing_df[missing_df["nuls"] > 0])

  # Détection dynamique des colonnes géographiques
  cp_col = next((c for c in df.columns if "postal" in c.lower()), None)
  insee_col = next(
      (
          c
          for c in df.columns
          if "insee" in c.lower() or c.lower() == "code_commune"
      ),
      None,
  )
  com_col = next(
      (
          c
          for c in df.columns
          if ("commune" in c.lower() or "ville" in c.lower())
          and "insee" not in c.lower()
          and "postal" not in c.lower()
      ),
      None,
  )

  # 3. ANALYSE DES FORMATS, ESPACES & VARIANTES DE CASSE
  print(f"\n--- 3. ANALYSE DES FORMATS & ANOMALIES DE SAISIE ---")

  # Audit Codes Postaux / Territoires
  if cp_col:
    cp_series = df[cp_col].dropna().astype(str).str.replace(r"\.0$", "", regex=True)
    has_decimal = df[cp_col].astype(str).str.contains(r"\.0$").sum()
    bad_length = (~cp_series.str.match(r"^\d{5}$")).sum()
    print(f" * [Code Postal] Formats décimaux (.0) : {has_decimal:,}")
    print(f" * [Code Postal] Formats non conformes (!= 5 chiffres) : {bad_length:,}")

  if insee_col:
    insee_series = (
        df[insee_col].dropna().astype(str).str.replace(r"\.0$", "", regex=True)
    )
    insee_decimal = df[insee_col].astype(str).str.contains(r"\.0$").sum()
    insee_bad_len = (~insee_series.str.match(r"^\d{5}$")).sum()
    print(f" * [Code Territoire] Formats décimaux (.0) : {insee_decimal:,}")
    print(
        f" * [Code Territoire] Formats non conformes (!= 5 chiffres) :"
        f" {insee_bad_len:,}"
    )

  # Audit Espaces superflus et Variations de Casse sur les colonnes textuelles
  for col in df.select_dtypes(include=["object"]).columns:
    s = df[col].dropna().astype(str).str.strip()

    # 1. Vérification des espaces superflus
    espaces = (
        df[col].dropna().astype(str).str.startswith(" ").sum()
        + df[col].dropna().astype(str).str.endswith(" ").sum()
    )
    if espaces > 0:
      print(f" * [Espaces] '{col}' contient {espaces:,} valeurs mal espacées.")

    # 2. Vérification des incohérences de casse
    lower_s = s.str.lower()
    variants = s.groupby(lower_s).nunique()
    bad_case_count = (variants > 1).sum()
    if bad_case_count > 0:
      exemples = list(variants[variants > 1].index)[:3]
      print(
          f" * [Casse hétérogène] '{col}' : {bad_case_count} termes ont des"
          f" variations de majuscules/minuscules (ex: {exemples})"
      )

  # Booléens textuels / Catégories courtes
  for col in df.select_dtypes(include=["object", "bool"]).columns:
    unique_vals = df[col].dropna().unique()
    str_vals = [str(v).lower().strip() for v in unique_vals]
    if len(unique_vals) <= 6 and any(
        val in ["true", "false", "1", "0", "oui", "non", "yes", "no", "faux"]
        for val in str_vals
    ):
      print(
          f" * [Booléen / Catégorie mixte] '{col}' -> Valeurs trouvées :"
          f" {unique_vals[:5]}"
      )

  # 4. VALEURS NUMÉRIQUES & EXTRÊMES
  print(
      f"\n--- 4. ANALYSE DES VALEURS NUMÉRIQUES & EXTRÊMES (Min / Max / Zéros)"
      f" ---"
  )
  num_cols = df.select_dtypes(include=[np.number]).columns
  for col in num_cols:
    s = df[col].dropna()
    if len(s) > 0:
      min_val = s.min()
      max_val = s.max()
      nb_zeros = (s == 0).sum()
      nb_neg = (s < 0).sum()

      alerte = ""
      if "puissance" in col.lower() and (min_val <= 0 or max_val > 500):
        alerte = " ⚠️ [ANOMALIE PUISSANCE SUSPECTE]"
      elif "lat" in col.lower() and not (-25 <= min_val and max_val <= 52):
        alerte = " ⚠️ [LATITUDE HORS ZONE]"
      elif "lon" in col.lower() and not (-65 <= min_val and max_val <= 56):
        alerte = " ⚠️ [LONGITUDE HORS ZONE]"

      print(
          f" - {col} | Min: {min_val} | Max: {max_val} | Zéros: {nb_zeros:,} |"
          f" Négatifs: {nb_neg:,}{alerte}"
      )

  # 5. DIAGNOSTIC DES DOUBLONS
  print(f"\n--- 5. DIAGNOSTIC DES DOUBLONS ---")
  print(f"- Lignes 100% identiques : {df.duplicated().sum():,}")
  for id_col in [
      "id_pdc_itinerance",
      "id_pdc_local",
      "id_station_itinerance",
      "code_insee_commune",
      "code_commune",
  ]:
    if id_col in df.columns:
      dup_id = df.duplicated(subset=[id_col]).sum()
      print(
          f"- Doublons sur '{id_col}' : {dup_id:,} ({dup_id / len(df) * 100:.2f}%)"
      )

  print("==================================================\n")

In [10]:
# 1. Diagnostic des colonnes cibles pour le dataset IRVE
colonnes_cibles_irve = [
    "id_station_itinerance",
    "id_pdc_itinerance",
    "nom_amenageur",
    "nom_operateur",
    "implantation_station",
    "condition_acces",
    "consolidated_latitude",
    "consolidated_longitude",
    "code_insee_commune",
    "consolidated_code_postal",
    "consolidated_commune",
    "puissance_nominale",
    "prise_type_2",
    "prise_type_combo_ccs",
    "prise_type_ef",
    "prise_type_chademo",
]

df_irve_cibles = pd.read_csv(
    "../data/raw/dataset_irve.csv",
    usecols=lambda c: c in colonnes_cibles_irve,
    low_memory=False,
)
diagnostic_complet(df_irve_cibles, nom_dataset="IRVE - Cibles")



=== DIAGNOSTIC COMPLET : IRVE - CIBLES ===

--- 1. VOLUMÉTRIE & STRUCTURE ---
Nombre de lignes : 226,963
Nombre de colonnes : 16

--- 2. SYNTHÈSE DES VALEURS MANQUANTES ---


,nuls,pourcentage
consolidated_code_postal,88511,39.00
consolidated_commune,70808,31.20
code_insee_commune,56573,24.93
nom_amenageur,1413,0.62
nom_operateur,474,0.21



--- 3. ANALYSE DES FORMATS & ANOMALIES DE SAISIE ---
 * [Code Postal] Formats décimaux (.0) : 138,452
 * [Code Postal] Formats non conformes (!= 5 chiffres) : 6,903
 * [Code Territoire] Formats décimaux (.0) : 0
 * [Code Territoire] Formats non conformes (!= 5 chiffres) : 354
 * [Espaces] 'nom_amenageur' contient 821 valeurs mal espacées.
 * [Casse hétérogène] 'nom_amenageur' : 38 termes ont des variations de majuscules/minuscules (ex: ['270 agency', 'as courtage', 'auberge saint walfrid'])
 * [Espaces] 'nom_operateur' contient 326 valeurs mal espacées.
 * [Casse hétérogène] 'nom_operateur' : 30 termes ont des variations de majuscules/minuscules (ex: ['bornevo', 'bp pulse', 'charge point'])
 * [Casse hétérogène] 'prise_type_ef' : 2 termes ont des variations de majuscules/minuscules (ex: ['false', 'true'])
 * [Casse hétérogène] 'prise_type_2' : 2 termes ont des variations de majuscules/minuscules (ex: ['false', 'true'])
 * [Casse hétérogène] 'prise_type_combo_ccs' : 2 termes ont des va

In [11]:
CHEMIN = "../data/raw/dataset_irve.csv"
ID = "id_pdc_itinerance"

df = pd.read_csv(CHEMIN, dtype=str, low_memory=False)
print(f"Dimensions : {df.shape}")

# %% 1. Les groupes atypiques (118 lignes pour un même ID, et 4 lignes)
taille_groupe = df.groupby(ID)[ID].transform("size")
cols_vue = [
    "datagouv_organization_or_owner", "nom_amenageur", "nom_station",
    "adresse_station", "puissance_nominale", "date_maj",
]
for t in (118, 4):
    ids = df.loc[taille_groupe == t, ID].unique()
    print(f"\n--- Groupe de {t} lignes : ID = {list(ids)} ---")
    display(df.loc[df[ID].isin(ids), [ID] + cols_vue].head(20))

# %% 2. Lignes de test / exemples de documentation
motifs = (
    r"exemple|soci[eé]t[eé] [xy]\b|r[eé]seau de recharge abc"
    r"|societe-(amenageur|operateur)\.com"
)
champs = ["nom_amenageur", "nom_operateur", "nom_enseigne",
          "contact_amenageur", "contact_operateur"]
masque_test = np.zeros(len(df), dtype=bool)
for c in champs:
    masque_test |= df[c].fillna("").str.contains(motifs, case=False, regex=True).to_numpy()
masque_test |= df["telephone_operateur"].fillna("").str.fullmatch(r"1{9}").to_numpy()

print(f"Lignes de test suspectes : {masque_test.sum():,}")
display(
    df.loc[masque_test, [ID, "nom_amenageur", "nom_operateur",
                         "datagouv_organization_or_owner"]].head(20)
)

# %% 3. Conflits entre sources (après normalisation)
# nunique() ignore les NaN : "valeur vs manquant" n'est pas un conflit,
# seul "valeur A vs valeur B" en est un.
cols_comparees = [
    "puissance_nominale", "nbre_pdc", "implantation_station", "condition_acces",
    "date_mise_en_service", "code_insee_commune",
    "prise_type_2", "prise_type_combo_ccs", "prise_type_ef", "prise_type_chademo",
]
dup = df[df.duplicated(subset=[ID], keep=False)].copy()
for c in cols_comparees:
    dup[c] = dup[c].astype("string").str.strip().str.lower()
dup["puissance_nominale"] = pd.to_numeric(dup["puissance_nominale"], errors="coerce")
dup["nbre_pdc"] = pd.to_numeric(dup["nbre_pdc"], errors="coerce")

conflits = dup.groupby(ID)[cols_comparees].nunique().gt(1).sum()
print("\n% d'IDs dupliqués avec une vraie divergence, par colonne :")
print((conflits / dup[ID].nunique() * 100).round(1).sort_values(ascending=False))

# %% 4. Qui publie les lignes en doublon ?
print("\nPublieurs les plus fréquents parmi les lignes dupliquées :")
print(dup["datagouv_organization_or_owner"].value_counts().head(15))

# %% 5. Répartition de la puissance (brut) et poids du palier 22 kW
p = pd.to_numeric(df["puissance_nominale"], errors="coerce")
print(f"\nPart de points à exactement 22 kW : {(p == 22).mean() * 100:.1f} %")
print(f"Points avec puissance > 500 : {(p > 500).sum():,}")
print(f"Points avec puissance >= 1000 : {(p >= 1000).sum():,}")
print("\nValeurs les plus fréquentes :")
print(p.value_counts().head(10))

Dimensions : (226963, 52)

--- Groupe de 118 lignes : ID = ['Non concerné'] ---


,id_pdc_itinerance,datagouv_organization_or_owner,nom_amenageur,nom_station,adresse_station,puissance_nominale,date_maj
226842,Non concerné,2ems,Aubade - Comptoir des Fers,Aubade - Comptoir des Fers,"1 Rue des Biches, 74100 Ville-la-Grand",22,2022-07-21
226843,Non concerné,4m-retail,4M RETAIL,Parking MAMBHOME,1 Impasse de Recouvrance 17100 SAINTES,22,2022-03-07
226844,Non concerné,a2bornes,SARL BEAUDRE BAUDOT,"BORNE SECURITEST, DOUVRES LA DELIVRANDE",Rue Jean Perrin 14440 Douvres-la-Délivrande,22,2022-07-06
226845,Non concerné,abe-electricite,La jabotte,La jabotte,13 Avenue Max Maurey 06160 Antibes,11,2021-05-07
226846,Non concerné,absolut,ABSOLUT,NANTEUIL,12 Av. Fridingen 77100 Nanteuil-lès-Meaux France,11,2023-02-11
226847,Non concerné,aeroport-de-lyon,Aeroports de lyon,Parking P4 électrique,"2 Rue de Finlande, 69124 Colombier-Saugnieu",3,2021-10-14
226848,Non concerné,agglomeration-montargoise-et-rives-du-loing,Agglomération Montargoise Et Rives du Loing,Parking Bassin du Loing,514 rue Raymond Lecerf,22,2023-03-22
226849,Non concerné,aintelec,ABSOLUT CONCEPT,Absolut Bougival,21 chemin du haut du trou martin,7,2022-02-24
226850,Non concerné,alc-car,ALC CAR,ALC CAR,2 RUE BRANLY 77400,11,2023-02-11
226851,Non concerné,apeira-technologies,DIDIER CHARTON-VACHET,"Borne, Domaine, Charton-Vachet",25 rue du May,22,2021-12-08



--- Groupe de 4 lignes : ID = ['FRLIBE3126085'] ---


,id_pdc_itinerance,datagouv_organization_or_owner,nom_amenageur,nom_station,adresse_station,puissance_nominale,date_maj
105512,FRLIBE3126085,bornevo,Résidence Carouge,Résidence Carouge,34 rue de l'Orge,7.4,2023-10-21
105513,FRLIBE3126085,charge-in,Residence Carouge,Residence Carouge,34 Rue de l’Orge 91220 Brétigny-sur-Orge,22,2023-11-10
105514,FRLIBE3126085,eoliberty,Eoliberty,Residence Carouge,34 Rue de lâÃÃ´Orge 91220 Brtigny-sur-Orge,22,2023-10-11
105515,FRLIBE3126085,greenea,Eoliberty,Residence Carouge,34 Rue de l‚Äö√Ñ√¥Orge 91220 Brétigny-sur-Orge,22,2023-10-11


C:\Users\cgboh\AppData\Local\Temp\ipykernel_12520\2625186249.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  masque_test |= df[c].fillna("").str.contains(motifs, case=False, regex=True).to_numpy()


Lignes de test suspectes : 6


,id_pdc_itinerance,nom_amenageur,nom_operateur,datagouv_organization_or_owner
101,ESZUNE1111ER7,Exemple invalide (id_station_itinerance et id_...,Société Y,totalenergies-marketing-france
3126,FRA68E680210015,Société X,Société Y,totalenergies-marketing-france
183338,FRSGAE1Z206301,SoPlug,SGA Industries,sga-industries
183339,FRSGAE1Z206302,SoPlug,SGA Industries,sga-industries
183340,FRSGAE1Z206303,SoPlug,SGA Industries,sga-industries
183341,FRSGAE1Z206304,SoPlug,SGA Industries,sga-industries



% d'IDs dupliqués avec une vraie divergence, par colonne :
implantation_station    41.2
nbre_pdc                33.5
code_insee_commune      18.0
date_mise_en_service    15.1
condition_acces         12.1
puissance_nominale      11.2
prise_type_2             8.8
prise_type_combo_ccs     8.1
prise_type_ef            7.5
prise_type_chademo       1.8
dtype: float64

Publieurs les plus fréquents parmi les lignes dupliquées :
datagouv_organization_or_owner
qualicharge                       52259
gireve-2                          12378
power-dot-france                   6556
lidl                               4823
totalenergies-marketing-france     4435
izivia                             4391
tesla-motors                       4380
engie-mobilites-electriques        3944
alize                              3105
electra                            2497
load-stations                      2077
e-totem                            2065
eco-movement                       1980
greenea                 

## Nettoyage IRVE v2

Objectif : obtenir **une ligne par point de charge**. Chaque étape est comptée dans un journal affiché sous la cellule ; le jeu intermédiaire `df_points` reste en mémoire et alimente la section suivante.

| Étape | Règle | Justification |
|---|---|---|
| Lecture | Toutes les colonnes lues en texte | Conserve les zéros initiaux des codes INSEE et postaux |
| Valeurs de remplissage | « Non concerné », « inconnu », « - », etc. converties en manquants dans les colonnes textuelles | Une valeur de remplissage n'est pas une information |
| Booléens | `True`, `true`, `TRUE`, `1` et équivalents ramenés à vrai/faux | 8 écritures différentes coexistent dans les colonnes de prises |
| Lignes de test | 2 identifiants de démonstration exclus (`ESZUNE1111ER7`, `FRA68E680210015`) ; 4 lignes SoPlug signalées par `flag_test_douteux` | Noms « Exemple invalide », « Société X / Société Y » |
| Identifiant réutilisé | `FRA68P680210015` exclu (3 lignes) | Le même identifiant désigne 3 points distincts publiés par 3 producteurs (Saint-Point, Dourdan, hostellerie) |
| Clé | `pdc_cle` = identifiant en majuscules ; identifiant absent ou « Non concerné » : clé unique par ligne | Les 118 lignes « Non concerné » sont 118 points distincts |
| Puissance | Valeurs en watts (7 360, 3 680, 1 840, 6 900, 12 800, 16 000, 22 000, 50 000, 160 000) divisées par 1 000 ; au-delà de 600 kW : `puissance_douteuse` ; 0 ou absente : `puissance_manquante` ; `puissance_brute` conservée | 600 kW est publié par 5 producteurs indépendants ; les valeurs au-delà sont non standard |
| Coordonnées | Conservées ; `coord_valide` et `coord_motif_invalide` (absente, zéro, latitude = longitude, hors emprise de la France et des outre-mer) | Les erreurs grossières sont marginales |
| Code INSEE | Retenu seulement si format valide et vérifié par le jeu consolidé ; `insee_statut` : `verifie`, `present_non_verifie`, `absent` | Des codes postaux et la valeur 99999 sont saisis comme codes INSEE |
| Arrondissements | Codes d'arrondissement de Paris (75101 à 75120), Marseille (13201 à 13216) et Lyon (69381 à 69389) rattachés à leur commune (75056, 13055, 69123) ; code d'origine conservé dans `code_arrondissement` ; `insee_statut` = `arrondissement_rattache` | Le jeu consolidé marque ces codes « non vérifiés » car son référentiel ne contient que les communes : 6 028 points concernés |
| Codes hors référentiel | Code « vérifié » absent du référentiel communal 2025 (`georef-france-commune`) : corrigé par le nom de la commune consolidée dans le département du code saisi si la correspondance est unique (`insee_statut` = `corrige_nom_commune`), sinon vidé (`a_geolocaliser`) ; code saisi conservé dans `code_insee_commune` ; origine du code dans `insee_source` | 4 647 points portent un code postal (67000, 86000, 75014…) ou un code d'ancienne commune marqué « vérifié » |
| Département | 2 premiers caractères du code INSEE, 3 pour les outre-mer (97x, 98x) | Distingue les 5 DROM |
| Commune des coordonnées | Commune 2025 dont le contour contient le point (`code_commune_geo`) ; hors contour : commune la plus proche si elle est à 200 m au plus (`distance_contour_m`) ; `coord_coherente` = la commune des coordonnées correspond au code ou au nom de commune publié par l'une des sources du point | Hors contour, les distances passent de 144 m à 1 461 m : 200 m sépare les points en bordure des erreurs de position |
| Dates | Avant 2000 ou dans le futur : vidées ; `date_ref` = `date_maj`, à défaut `last_modified` | Dates impossibles |
| Dédoublonnage | Une ligne par `pdc_cle`, choisie dans l'ordre : position cohérente, géographie vérifiée, puissance renseignée, complétude des colonnes utiles, récence, identifiant de ressource ; tri stable | Un même point est publié par plusieurs producteurs ; 792 points ont des sources distantes de plus de 1 km (latitude = longitude, signe ou chiffre manquant dans une source) |
| Traçabilité | `nb_sources`, `source_retenue`, `conflit_puissance`, `conflit_implantation`, `conflit_nbre_pdc` | Mesure les divergences entre producteurs |

Hors périmètre de cette étape : contrôle des codes INSEE par la position (section suivante), jointure avec la population, classes de puissance, harmonisation des noms d'aménageurs et du vocabulaire d'implantation.


In [ ]:
# ===================================================================
# NETTOYAGE IRVE v2 : une ligne par point de charge, aucune suppression, chaque étape comptée dans un journal
# ===================================================================
CHEMIN_BRUT = Path("../data/raw/dataset_irve.csv")
CHEMIN_REFERENTIEL = Path("../data/raw/referentiel_communes.csv")

# --- Paramètres ---------------------------------------------------
PLACEHOLDERS = {
    "non concerné", "non concerne", "non renseigné", "non renseigne",
    "inconnu", "-", "n/a", "na", "nan", "null", "none",
}
COLONNES_NON_TEXTE = {
    "puissance_nominale", "nbre_pdc", "consolidated_latitude",
    "consolidated_longitude", "date_mise_en_service", "date_maj",
    "last_modified", "created_at",
}
COLONNES_BOOLEENNES = [
    "prise_type_ef", "prise_type_2", "prise_type_combo_ccs", "prise_type_chademo",
    "prise_type_autre", "gratuit", "paiement_acte", "paiement_cb",
    "paiement_autre", "reservation", "station_deux_roues", "cable_t2_attache",
    "consolidated_is_lon_lat_correct", "consolidated_is_code_insee_verified",
    "consolidated_is_code_insee_modified",
]
IDS_TEST_CERTAINS = {"ESZUNE1111ER7", "FRA68E680210015"}
IDS_TEST_DOUTEUX = {f"FRSGAE1Z20630{i}" for i in range(1, 5)}
IDS_REUTILISES = {"FRA68P680210015"}
DISTANCE_MAX_CONTOUR_M = 200
PUISSANCES_EN_WATTS = {7360, 3680, 1840, 6900, 12800, 16000, 22000, 50000, 160000}
PUISSANCE_MAX_KW = 600
DATE_MIN = pd.Timestamp("2000-01-01")

# Emprises approximatives (lat_min, lat_max, lon_min, lon_max)
EMPRISES = {
    "Métropole et Corse": (41.0, 51.5, -5.5, 10.0),
    "Guadeloupe": (15.8, 16.6, -61.9, -60.9),
    "Martinique": (14.3, 15.0, -61.3, -60.7),
    "Guyane": (2.0, 6.0, -54.7, -51.5),
    "La Réunion": (-21.5, -20.8, 55.2, 55.9),
    "Mayotte": (-13.1, -12.6, 45.0, 45.4),
    "Saint-Pierre-et-Miquelon": (46.7, 47.2, -56.5, -56.0),
    "Saint-Barthélemy et Saint-Martin": (17.8, 18.2, -63.2, -62.7),
    "Nouvelle-Calédonie": (-23.0, -19.5, 163.5, 168.2),
    "Polynésie française": (-28.0, -7.0, -155.0, -134.0),
    "Wallis-et-Futuna": (-14.4, -13.1, -178.3, -176.0),
}

journal = []


def journaliser(etape, avant, apres, concernees, detail):
  journal.append({
      "etape": etape, "lignes_avant": avant, "lignes_apres": apres,
      "lignes_concernees": int(concernees), "detail": detail,
  })
  print(f"[{etape}] {avant:,} -> {apres:,} lignes | concernées : {int(concernees):,} | {detail}")


# --- 1. Lecture intégrale en texte --------------------------------
df = pd.read_csv(CHEMIN_BRUT, dtype=str, keep_default_na=False, low_memory=False)
n0 = len(df)
journaliser("1_lecture", n0, n0, 0, f"{df.shape[1]} colonnes lues en texte")
for c in df.columns:
  df[c] = df[c].str.strip()
df = df.replace("", np.nan)

# --- 2. Placeholders -> manquants (colonnes textuelles) -----------
colonnes_texte = [c for c in df.columns
                  if c not in COLONNES_NON_TEXTE and c not in COLONNES_BOOLEENNES]
nb_ph = 0
for c in colonnes_texte:
  masque = df[c].str.lower().isin(PLACEHOLDERS)
  nb_ph += masque.sum()
  df.loc[masque, c] = np.nan
journaliser("2_placeholders", n0, n0, nb_ph,
            "valeurs de remplissage converties en manquants (cellules)")

# --- 3. Booléens harmonisés ---------------------------------------
correspondance = {"true": True, "1": True, "false": False, "0": False}
for c in COLONNES_BOOLEENNES:
  df[c] = df[c].str.lower().map(correspondance).astype("boolean")
journaliser("3_booleens", n0, n0, 0,
            f"{len(COLONNES_BOOLEENNES)} colonnes ramenées à vrai/faux")

# --- 4. Lignes de test --------------------------------------------
id_maj = df["id_pdc_itinerance"].str.upper()
masque_test = id_maj.isin(IDS_TEST_CERTAINS)
df = df.loc[~masque_test].copy()
id_maj = id_maj.loc[df.index]
journaliser("4_lignes_test", n0, len(df), masque_test.sum(),
            "lignes de test certaines exclues")
masque_reutilise = id_maj.isin(IDS_REUTILISES)
avant = len(df)
df = df.loc[~masque_reutilise].copy()
id_maj = id_maj.loc[df.index]
journaliser("4c_id_reutilise", avant, len(df), masque_reutilise.sum(),
            "identifiant partagé par des points distincts : lignes exclues")
df["flag_test_douteux"] = id_maj.isin(IDS_TEST_DOUTEUX)
journaliser("4b_test_douteux", len(df), len(df), df["flag_test_douteux"].sum(),
            "lignes douteuses conservées et signalées")

# --- 5. Clé du point de charge ------------------------------------
sans_id = id_maj.isna()
df["pdc_cle"] = id_maj.where(~sans_id, "SANS_ID_" + df.index.astype(str))
journaliser("5_cle", len(df), len(df), sans_id.sum(),
            "identifiants absents ou 'Non concerné' : clé unique par ligne")

# --- 6. Puissance -------------------------------------------------
df["puissance_brute"] = df["puissance_nominale"]
p = pd.to_numeric(df["puissance_nominale"].str.replace(",", "."), errors="coerce")
en_watts = p.isin(PUISSANCES_EN_WATTS)
p = p.where(~en_watts, p / 1000)
df["puissance_convertie_w"] = en_watts
df["puissance_manquante"] = p.isna() | (p == 0)
df["puissance_douteuse"] = p > PUISSANCE_MAX_KW
df["puissance_kw"] = p.where(~df["puissance_manquante"] & ~df["puissance_douteuse"])
journaliser("6a_puissance_watts", len(df), len(df), en_watts.sum(),
            "valeurs en watts divisées par 1 000")
journaliser("6b_puissance_douteuse", len(df), len(df), df["puissance_douteuse"].sum(),
            f"> {PUISSANCE_MAX_KW} kW après conversion : puissance_kw vide")
journaliser("6c_puissance_manquante", len(df), len(df), df["puissance_manquante"].sum(),
            "puissance absente ou nulle : puissance_kw vide")

# --- 7. Coordonnées -----------------------------------------------
lat = pd.to_numeric(df["consolidated_latitude"], errors="coerce")
lon = pd.to_numeric(df["consolidated_longitude"], errors="coerce")
dans_emprise = pd.Series(False, index=df.index)
for la_min, la_max, lo_min, lo_max in EMPRISES.values():
  dans_emprise |= lat.between(la_min, la_max) & lon.between(lo_min, lo_max)
motif = pd.Series(pd.NA, index=df.index, dtype="string")
motif = motif.mask(~dans_emprise, "hors_emprise")
motif = motif.mask(lat == lon, "latitude_egale_longitude")
motif = motif.mask((lat == 0) | (lon == 0), "zero")
motif = motif.mask(lat.isna() | lon.isna(), "absente")
df["latitude"], df["longitude"] = lat, lon
df["coord_valide"] = motif.isna()
df["coord_motif_invalide"] = motif
journaliser("7_coordonnees", len(df), len(df), (~df["coord_valide"]).sum(),
            "coordonnées invalides signalées (conservées)")

# --- 8. Code INSEE et département ---------------------------------
insee = df["code_insee_commune"].str.upper()
insee = insee.where(~insee.str.fullmatch(r"\d{4}", na=False), "0" + insee)
format_ok = insee.str.fullmatch(r"\d{5}|2[AB]\d{3}", na=False) & (insee != "99999")
verifie = df["consolidated_is_code_insee_verified"].fillna(False).astype(bool)
# Arrondissements municipaux de Paris, Marseille et Lyon : rattachés à leur commune
arrondissement = insee.str.fullmatch(
    r"751(0[1-9]|1\d|20)|132(0[1-9]|1[0-6])|6938[1-9]", na=False)
COMMUNES_A_ARRONDISSEMENTS = {"751": "75056", "132": "13055", "693": "69123"}
df["code_arrondissement"] = insee.where(arrondissement)
df["insee_statut"] = np.select(
    [arrondissement, format_ok & verifie, insee.notna()],
    ["arrondissement_rattache", "verifie", "present_non_verifie"], default="absent")
df["code_insee"] = insee.where(df["insee_statut"] == "verifie")
df.loc[arrondissement, "code_insee"] = insee[arrondissement].str[:3].map(
    COMMUNES_A_ARRONDISSEMENTS)

# Codes « vérifiés » absents du référentiel communal (codes postaux, anciennes
# communes) : corrigés par le nom de la commune consolidée dans le même département
# lorsque la correspondance est unique, sinon vidés en attente de géolocalisation


def normaliser_nom(nom):
  nom = unicodedata.normalize("NFKD", str(nom)).encode("ascii", "ignore").decode().lower()
  nom = re.sub(r"st", "saint", nom)
  nom = re.sub(r"ste", "sainte", nom)
  return re.sub(r"[^a-z0-9]", "", nom)


referentiel = pd.read_csv(CHEMIN_REFERENTIEL, sep=";", dtype=str,
                          usecols=["com_code", "com_name", "dep_code", "reg_name",
                                   "geo_point_2d", "geo_shape"])
referentiel["cle"] = referentiel["dep_code"] + "|" + referentiel["com_name"].map(normaliser_nom)
code_par_nom = referentiel.drop_duplicates("cle", keep=False).set_index("cle")["com_code"]
hors_referentiel = (df["insee_statut"] == "verifie") & ~df["code_insee"].isin(
    set(referentiel["com_code"]))
dep_saisi = np.where(df["code_insee"].str[:2].isin(["97", "98"]),
                     df["code_insee"].str[:3], df["code_insee"].str[:2])
cle = pd.Series(dep_saisi, index=df.index) + "|" + df["consolidated_commune"].map(
    normaliser_nom)
code_corrige = cle.where(hors_referentiel & df["consolidated_commune"].notna()).map(code_par_nom)
corrige = hors_referentiel & code_corrige.notna()
df.loc[corrige, "code_insee"] = code_corrige[corrige]
df.loc[corrige, "insee_statut"] = "corrige_nom_commune"
df.loc[hors_referentiel & ~corrige, "code_insee"] = np.nan
df.loc[hors_referentiel & ~corrige, "insee_statut"] = "a_geolocaliser"
df["insee_source"] = df["insee_statut"].map({
    "verifie": "declare", "arrondissement_rattache": "declare",
    "corrige_nom_commune": "corrige_nom_commune"})
journaliser("8_insee_hors_referentiel", len(df), len(df), hors_referentiel.sum(),
            "codes vérifiés absents du référentiel communal 2025")
df["code_departement"] = np.where(
    df["code_insee"].str[:2].isin(["97", "98"]),
    df["code_insee"].str[:3], df["code_insee"].str[:2])
df["code_departement"] = df["code_departement"].where(df["code_insee"].notna())
for statut, n in df["insee_statut"].value_counts().items():
  journaliser(f"8_insee_{statut}", len(df), len(df), n, "statut du code INSEE")

# --- 9. Dates -----------------------------------------------------
date_jour = pd.Timestamp.today().normalize()
for c in ["date_mise_en_service", "date_maj"]:
  d = pd.to_datetime(df[c], errors="coerce", format="%Y-%m-%d")
  hors_bornes = (d < DATE_MIN) | (d > date_jour)
  df[c] = d.where(~hors_bornes)
  journaliser(f"9_{c}", len(df), len(df), hors_bornes.sum(),
              f"dates avant 2000 ou futures (après le {date_jour.date()}) vidées")
last_mod = pd.to_datetime(df["last_modified"], errors="coerce", utc=True)
df["date_ref"] = df["date_maj"].fillna(last_mod.dt.tz_localize(None).dt.normalize())

# --- 9b. Commune des coordonnées (contours communaux 2025) ---------
communes = gpd.GeoDataFrame(
    referentiel[["com_code", "com_name"]],
    geometry=referentiel["geo_shape"].map(lambda s: shape(json.loads(s))), crs=4326)
avec_coord = df["coord_valide"]
points = gpd.GeoDataFrame(
    index=df.index[avec_coord],
    geometry=gpd.points_from_xy(df.loc[avec_coord, "longitude"],
                                df.loc[avec_coord, "latitude"]), crs=4326)
dans = gpd.sjoin(points, communes[["com_code", "geometry"]], how="left", predicate="within")
dans = dans[~dans.index.duplicated()]
df["code_commune_geo"] = dans["com_code"].reindex(df.index)
df["distance_contour_m"] = np.where(df["code_commune_geo"].notna(), 0.0, np.nan)
hors_contour = dans.index[dans["com_code"].isna()]
proches = gpd.sjoin_nearest(points.loc[hors_contour].to_crs(3857),
                            communes[["com_code", "geometry"]].to_crs(3857),
                            how="left", distance_col="distance")
proches = proches[~proches.index.duplicated()]
distance_m = proches["distance"] * np.cos(np.radians(df.loc[proches.index, "latitude"]))
df.loc[proches.index, "distance_contour_m"] = distance_m
retenus = proches.index[distance_m <= DISTANCE_MAX_CONTOUR_M]
df.loc[retenus, "code_commune_geo"] = proches.loc[retenus, "com_code"]
journaliser("9b_commune_contour", len(df), len(df), len(dans) - len(hors_contour),
            "points situés dans un contour communal")
journaliser("9b_commune_proche", len(df), len(df), len(retenus),
            f"hors contour, commune la plus proche à {DISTANCE_MAX_CONTOUR_M} m au plus")
journaliser("9b_sans_commune", len(df), len(df), df["code_commune_geo"].isna().sum(),
            "coordonnées invalides ou à plus de 200 m de toute commune du référentiel")

# Position cohérente : la commune des coordonnées est celle du code ou du nom de commune
# publiés par la ligne ou par une autre source du même point
nom_par_code = referentiel.set_index("com_code")["com_name"].map(normaliser_nom)
nom_consolide = df["consolidated_commune"].map(normaliser_nom).where(
    df["consolidated_commune"].notna())
code_groupe = df.groupby("pdc_cle")["code_insee"].transform("first")
nom_groupe = nom_consolide.groupby(df["pdc_cle"]).transform("first")
nom_geo = df["code_commune_geo"].map(nom_par_code)
df["coord_coherente"] = df["code_commune_geo"].notna() & (
    (df["code_commune_geo"] == df["code_insee"]) | (df["code_commune_geo"] == code_groupe)
    | (nom_geo == nom_consolide) | (nom_geo == nom_groupe))
journaliser("9b_coord_coherente", len(df), len(df), df["coord_coherente"].sum(),
            "lignes dont la position confirme le code ou le nom de commune d'une source du point")

# Date de mise en service absente de la ligne : reprise d'une autre source du même point
date_autre_source = df.groupby("pdc_cle")["date_mise_en_service"].transform("first")
date_reprise = df["date_mise_en_service"].isna() & date_autre_source.notna()
df["date_mise_en_service"] = df["date_mise_en_service"].fillna(date_autre_source)
journaliser("9c_date_autre_source", len(df), len(df), date_reprise.sum(),
            "date de mise en service reprise d'une autre source du même point")

# --- 10. Dédoublonnage par score ----------------------------------
colonnes_utiles = [
    "nom_amenageur", "nom_operateur", "id_station_itinerance", "adresse_station",
    "implantation_station", "condition_acces", "nbre_pdc", "puissance_kw",
    "prise_type_2", "prise_type_combo_ccs", "prise_type_chademo", "prise_type_ef",
    "date_mise_en_service", "code_insee",
]
df["score_geo"] = (df["insee_statut"].isin(
    ["verifie", "arrondissement_rattache", "corrige_nom_commune"]).astype(int)
                   + df["coord_valide"].astype(int))
df["score_puissance"] = df["puissance_kw"].notna().astype(int)
df["score_completude"] = df[colonnes_utiles].notna().sum(axis=1)

groupe = df.groupby("pdc_cle")
df["nb_sources"] = groupe["pdc_cle"].transform("size")
conflits = pd.DataFrame({
    "conflit_puissance": groupe["puissance_kw"].transform("nunique"),
    "conflit_implantation": df.assign(
        v=df["implantation_station"].str.lower()).groupby("pdc_cle")["v"].transform("nunique"),
    "conflit_nbre_pdc": pd.to_numeric(df["nbre_pdc"], errors="coerce")
    .groupby(df["pdc_cle"]).transform("nunique"),
}) > 1
df[conflits.columns] = conflits

df = df.sort_values(
    ["pdc_cle", "coord_coherente", "score_geo", "score_puissance", "score_completude",
     "date_ref", "datagouv_resource_id"],
    ascending=[True, False, False, False, False, False, True],
    kind="mergesort", na_position="last")
avant = len(df)
df = df.drop_duplicates(subset="pdc_cle", keep="first")
df["source_retenue"] = df["datagouv_organization_or_owner"]
journaliser("10_dedoublonnage", avant, len(df), avant - len(df),
            "une ligne retenue par point (position cohérente, géographie, puissance, complétude, récence)")
for c in conflits.columns:
  journaliser(f"10_{c}", len(df), len(df), df[c].sum(),
              "points dont les sources divergent")

# --- 11. Sélection des colonnes et sauvegarde ---------------------
colonnes_finales = [
    "pdc_cle", "id_pdc_itinerance", "id_pdc_local", "id_station_itinerance",
    "nom_station", "adresse_station", "nom_amenageur", "siren_amenageur",
    "nom_operateur", "nom_enseigne", "implantation_station", "condition_acces",
    "nbre_pdc", "puissance_brute", "puissance_kw", "puissance_convertie_w",
    "puissance_manquante", "puissance_douteuse",
    "prise_type_ef", "prise_type_2", "prise_type_combo_ccs", "prise_type_chademo",
    "prise_type_autre", "gratuit", "accessibilite_pmr", "horaires",
    "latitude", "longitude", "coord_valide", "coord_motif_invalide",
    "code_commune_geo", "distance_contour_m", "coord_coherente",
    "consolidated_is_lon_lat_correct",
    "code_insee_commune", "code_insee", "code_arrondissement", "insee_statut", "insee_source",
    "code_departement",
    "consolidated_commune", "consolidated_code_postal",
    "date_mise_en_service", "date_maj", "date_ref",
    "flag_test_douteux", "nb_sources", "source_retenue", "datagouv_resource_id",
    "conflit_puissance", "conflit_implantation", "conflit_nbre_pdc",
]
df_points = df[colonnes_finales].reset_index(drop=True)
journaliser("11_sortie", len(df_points), len(df_points), 0,
            f"{df_points.shape[1]} colonnes conservées")

print(f"\nPoints de charge : {len(df_points):,}")
print(df_points["insee_statut"].value_counts(normalize=True).mul(100).round(1))
print(f"Coordonnées valides : {df_points['coord_valide'].mean() * 100:.1f} %")
print(f"Puissance renseignée : {df_points['puissance_kw'].notna().mean() * 100:.1f} %")


[1_lecture] 226,963 -> 226,963 lignes | concernées : 0 | 52 colonnes lues en texte
[2_placeholders] 226,963 -> 226,963 lignes | concernées : 54,387 | valeurs de remplissage converties en manquants (cellules)
[3_booleens] 226,963 -> 226,963 lignes | concernées : 0 | 15 colonnes ramenées à vrai/faux
[4_lignes_test] 226,963 -> 226,959 lignes | concernées : 4 | lignes de test certaines exclues
[4c_id_reutilise] 226,959 -> 226,956 lignes | concernées : 3 | identifiant partagé par des points distincts : lignes exclues
[4b_test_douteux] 226,956 -> 226,956 lignes | concernées : 4 | lignes douteuses conservées et signalées
[5_cle] 226,956 -> 226,956 lignes | concernées : 118 | identifiants absents ou 'Non concerné' : clé unique par ligne
[6a_puissance_watts] 226,956 -> 226,956 lignes | concernées : 883 | valeurs en watts divisées par 1 000
[6b_puissance_douteuse] 226,956 -> 226,956 lignes | concernées : 161 | > 600 kW après conversion : puissance_kw vide
[6c_puissance_manquante] 226,956 -> 226,

## Rattachement géographique des points

Objectif : donner à chaque point un code INSEE de commune **contrôlé par sa position**, à partir des contours communaux 2025 (`georef-france-commune`), puis produire le jeu propre `data/processed/irve_points_clean.csv`, sans aucune valeur manquante.

| Cas | Règle | Justification |
|---|---|---|
| Code fiable (vérifié, arrondissement, corrigé par nom) et position dans la même commune | Conservé, `controle_geo` = `coherent` | Deux sources concordent |
| Code fiable, position dans une autre commune, nom de commune = commune des coordonnées | Code remplacé par la commune des coordonnées (`corrige_geo`) | Le code publié est un code postal qui est aussi le code INSEE d'une autre commune (ex. 77500 : code postal de Chelles, code INSEE de Villebéon) |
| Code fiable, position dans une autre commune, nom de commune = commune du code | Code conservé ; `limite_communale` si le point est à 500 m au plus de la commune déclarée, sinon `coord_suspectes` | Grands sites à cheval sur deux communes (aéroports) ; au-delà, la position est fausse |
| Code fiable, aucune des deux communes ne correspond au nom | Code conservé, `indetermine` | Pas d'arbitrage possible |
| Code absent, non vérifié ou non retrouvé | Commune des coordonnées (`geolocalise`) ; aucune commune à 200 m : code vide (`sans_commune`) | 99,9 % des points sont dans un contour communal ; position et nom concordent dans 96,7 % des cas sur les codes fiables |


**Jeu propre final**

| Étape | Règle | Justification |
|---|---|---|
| Coordonnées suspectes | Position remplacée par le centre de la commune (référentiel 2025) | La commune est confirmée par le code et par le nom publiés ; seule la position est fausse. Les supprimer laissait 518 communes (572 180 habitants) sans aucune borne : faux déserts |
| Puissance inconnue | Médiane des points ayant la même combinaison de prises (type 2 seule : 22 kW) | Eco-movement publie 0 kW pour 2 308 points réels |
| Suppression | Commune non établie (indéterminée, non contrôlée, sans commune), lignes de test douteuses, puissance toujours inconnue | Donnée inexploitable pour la cartographie ou l'analyse de puissance |
| Aménageur / opérateur | Quand un seul des deux est publié, il tient les deux rôles ; sans aucun des deux : ligne supprimée | Un acteur unique publie souvent seul sa borne |
| Station | Identifiant absent : clé construite sur la position (`LOC_latitude_longitude`) | Les points situés au même emplacement forment une station |
| Nom de station | Absent : adresse reprise | Le nom sert d'étiquette |
| Libellés | Encodage corrompu (« Accčs libre », « ŕ usage public ») ramené au libellé correct | Même modalité écrite différemment |

Colonnes conservées (20) : `id_pdc`, `id_station`, `nom_station`, `adresse_station`, `nom_amenageur`, `nom_operateur`, `implantation_station`, `condition_acces`, `puissance_kw`, `prise_type_2`, `prise_type_combo_ccs`, `prise_type_chademo`, `prise_type_ef`, `latitude`, `longitude`, `code_insee`, `nom_commune`, `code_departement`, `region`, `date_mise_en_service`. Le nom de commune et la région viennent du référentiel 2025 et servent à contrôler la jointure avec la population.

`date_mise_en_service` est la seule colonne incomplète : elle est reprise d'une autre source du même point quand la ligne retenue ne la publie pas, mais plusieurs producteurs ne la publient jamais (gireve-2, alize, indigo-group). Les analyses de tendance portent sur les points datés et ce biais de publication doit être mentionné.


In [13]:
# ===================================================================
# RATTACHEMENT GÉOGRAPHIQUE ET JEU PROPRE FINAL
# Utilise df_points, communes, nom_par_code et normaliser_nom (cellule précédente)
# ===================================================================
CHEMIN_CLEAN = Path("../data/processed/irve_points_clean.csv")
DISTANCE_LIMITE_COMMUNALE_M = 500

journal = []
geo = df_points.copy()
n = len(geo)
journaliser("1_entree", n, n, 0, "points issus du nettoyage v2")

statuts_fiables = ["verifie", "arrondissement_rattache", "corrige_nom_commune"]
statuts_cibles = ["absent", "present_non_verifie", "a_geolocaliser"]
fiable = geo["insee_statut"].isin(statuts_fiables)
cible = geo["insee_statut"].isin(statuts_cibles)
a_commune_geo = geo["code_commune_geo"].notna()

# --- 2. Codes fiables en désaccord avec la position (règle D) ------
desaccord = fiable & a_commune_geo & (geo["code_commune_geo"] != geo["code_insee"])
nom_consolide = geo["consolidated_commune"].map(normaliser_nom).where(
    geo["consolidated_commune"].notna())
code_faux = desaccord & (nom_consolide == geo["code_commune_geo"].map(nom_par_code))
code_confirme = desaccord & ~code_faux & (nom_consolide == geo["code_insee"].map(nom_par_code))
indetermine = desaccord & ~code_faux & ~code_confirme

# Distance entre le point et la commune déclarée, pour les codes confirmés par le nom
idx = geo.index[code_confirme]
contours = communes.set_index("com_code")["geometry"]
point_m = gpd.GeoSeries(gpd.points_from_xy(geo.loc[idx, "longitude"], geo.loc[idx, "latitude"]),
                        index=idx, crs=4326).to_crs(3857)
contour_m = gpd.GeoSeries(contours.loc[geo.loc[idx, "code_insee"]].values,
                          index=idx, crs=4326).to_crs(3857)
geo["distance_commune_declaree_m"] = np.nan
geo.loc[idx, "distance_commune_declaree_m"] = (
    point_m.distance(contour_m) * np.cos(np.radians(geo.loc[idx, "latitude"])))
limite = code_confirme & (geo["distance_commune_declaree_m"] <= DISTANCE_LIMITE_COMMUNALE_M)
suspect = code_confirme & ~limite

geo["controle_geo"] = np.select(
    [fiable & a_commune_geo & ~desaccord, code_faux, limite, suspect, indetermine,
     fiable & ~a_commune_geo],
    ["coherent", "code_corrige_geo", "limite_communale", "coord_suspectes", "indetermine",
     "non_controle"], default=None)
geo.loc[code_faux, "code_insee"] = geo.loc[code_faux, "code_commune_geo"]
geo.loc[code_faux, "code_arrondissement"] = np.nan
geo.loc[code_faux, ["insee_statut", "insee_source"]] = "corrige_geo"
geo["coord_suspectes"] = suspect
for valeur in ["coherent", "code_corrige_geo", "limite_communale", "coord_suspectes",
               "indetermine", "non_controle"]:
  journaliser(f"2_fiable_{valeur}", n, n, (geo["controle_geo"] == valeur).sum(),
              "codes fiables : contrôle par la position")

# --- 3. Codes absents ou non fiables : commune des coordonnées (règle B)
geolocalise = cible & a_commune_geo
geo.loc[geolocalise, "code_insee"] = geo.loc[geolocalise, "code_commune_geo"]
geo.loc[geolocalise, ["insee_statut", "insee_source"]] = "geolocalise"
geo.loc[geolocalise, "controle_geo"] = "geolocalise"
geo.loc[cible & ~a_commune_geo, "controle_geo"] = "sans_commune"
journaliser("3_geolocalise", n, n, geolocalise.sum(),
            "code absent ou non fiable remplacé par la commune des coordonnées")
journaliser("3_sans_commune", n, n, (cible & ~a_commune_geo).sum(),
            "code absent ou non fiable, aucune commune à 200 m : code vide")

# --- 4. Département ------------------------------------------------
geo["code_departement"] = np.where(
    geo["code_insee"].str[:2].isin(["97", "98"]),
    geo["code_insee"].str[:3], geo["code_insee"].str[:2])
geo["code_departement"] = geo["code_departement"].where(geo["code_insee"].notna())

# --- 5. Correction des faux déserts --------------------------------
# Coordonnées suspectes : la commune est confirmée par le code et par le nom publiés,
# seule la position est fausse ; elle est remplacée par le centre de la commune
centre = referentiel.set_index("com_code")["geo_point_2d"].str.split(",", expand=True).astype(float)
suspect = geo["controle_geo"] == "coord_suspectes"
geo.loc[suspect, "latitude"] = geo.loc[suspect, "code_insee"].map(centre[0])
geo.loc[suspect, "longitude"] = geo.loc[suspect, "code_insee"].map(centre[1])
journaliser("5_coord_centre_commune", n, n, suspect.sum(),
            "coordonnées suspectes remplacées par le centre de la commune")

# Puissance inconnue : médiane des points ayant la même combinaison de prises
prises = ["prise_type_2", "prise_type_combo_ccs", "prise_type_chademo", "prise_type_ef"]
combinaison = geo[prises].astype(str).agg("|".join, axis=1)
mediane = geo["puissance_kw"].groupby(combinaison).median()
sans_puissance = geo["puissance_kw"].isna()
geo.loc[sans_puissance, "puissance_kw"] = combinaison[sans_puissance].map(mediane)
journaliser("5_puissance_mediane_prises", n, n, (sans_puissance & geo["puissance_kw"].notna()).sum(),
            "puissance inconnue : médiane des points ayant les mêmes prises")

# --- 5b. Suppression des points inexploitables ---------------------
motifs_suppression = {
    "commune_non_etablie": geo["controle_geo"].isin(["indetermine", "non_controle", "sans_commune"]),
    "test_douteux": geo["flag_test_douteux"].astype(bool),
    "puissance_inconnue": geo["puissance_kw"].isna(),
}
for motif, masque in motifs_suppression.items():
  avant = len(geo)
  geo = geo.loc[~masque.loc[geo.index]]
  journaliser(f"5b_{motif}", avant, len(geo), avant - len(geo), "lignes supprimées")

# --- 6. Complétion des valeurs manquantes --------------------------
# Aménageur et opérateur : quand un seul est publié, il tient les deux rôles
operateur_vide = geo["nom_operateur"].isna() & geo["nom_amenageur"].notna()
amenageur_vide = geo["nom_amenageur"].isna() & geo["nom_operateur"].notna()
geo.loc[operateur_vide, "nom_operateur"] = geo.loc[operateur_vide, "nom_amenageur"]
geo.loc[amenageur_vide, "nom_amenageur"] = geo.loc[amenageur_vide, "nom_operateur"]
journaliser("6_operateur_complete", len(geo), len(geo), operateur_vide.sum(),
            "opérateur absent : aménageur repris")
journaliser("6_amenageur_complete", len(geo), len(geo), amenageur_vide.sum(),
            "aménageur absent : opérateur repris")
avant = len(geo)
geo = geo.loc[geo["nom_amenageur"].notna()]
journaliser("6_sans_acteur", avant, len(geo), avant - len(geo),
            "ni aménageur ni opérateur : lignes supprimées")

# Station absente : les points situés au même emplacement forment une station
sans_station = geo["id_station_itinerance"].isna()
geo.loc[sans_station, "id_station_itinerance"] = (
    "LOC_" + geo.loc[sans_station, "latitude"].round(5).astype(str)
    + "_" + geo.loc[sans_station, "longitude"].round(5).astype(str))
journaliser("6_station_completee", len(geo), len(geo), sans_station.sum(),
            "identifiant de station absent : clé construite sur la position")

sans_nom = geo["nom_station"].isna()
geo.loc[sans_nom, "nom_station"] = geo.loc[sans_nom, "adresse_station"]
journaliser("6_nom_station_complete", len(geo), len(geo), sans_nom.sum(),
            "nom de station absent : adresse reprise")

# Libellés à l'encodage corrompu (« Accčs libre », « ŕ usage public ») : ramenés au
# libellé le plus fréquent qui a les mêmes lettres non accentuées
for colonne in ["implantation_station", "condition_acces"]:
  squelette = geo[colonne].str.lower().str.replace(r"[^a-z]", "", regex=True)
  libelle = geo[colonne].groupby(squelette).agg(lambda s: s.value_counts().index[0])
  corrige = squelette.map(libelle)
  journaliser(f"6_libelles_{colonne}", len(geo), len(geo), (corrige != geo[colonne]).sum(),
              "libellés à l'encodage corrompu corrigés")
  geo[colonne] = corrige

# --- 7. Jeu propre final -------------------------------------------
# Nom de commune et région du référentiel 2025, pour contrôler la jointure population
noms = referentiel.set_index("com_code")
geo["nom_commune"] = geo["code_insee"].map(noms["com_name"])
geo["region"] = geo["code_insee"].map(noms["reg_name"])
df_clean = geo[[
    "pdc_cle", "id_station_itinerance", "nom_station", "adresse_station",
    "nom_amenageur", "nom_operateur", "implantation_station", "condition_acces",
    "puissance_kw", "prise_type_2", "prise_type_combo_ccs", "prise_type_chademo",
    "prise_type_ef", "latitude", "longitude", "code_insee", "nom_commune",
    "code_departement", "region", "date_mise_en_service",
]].rename(columns={"pdc_cle": "id_pdc", "id_station_itinerance": "id_station"})
df_clean = df_clean.reset_index(drop=True)
# Seule exception : la date de mise en service, non publiée par certains producteurs
manquants = df_clean.drop(columns="date_mise_en_service").isna().sum()
if manquants.any():
  raise ValueError(f"Valeurs manquantes restantes : {manquants[manquants > 0].to_dict()}")
taux_date = df_clean["date_mise_en_service"].notna().mean() * 100
journaliser("7_sortie", len(df_points), len(df_clean), len(df_points) - len(df_clean),
            f"jeu propre : {df_clean.shape[1]} colonnes, aucune valeur manquante hors date"
            f" de mise en service ({taux_date:.1f} % renseignée)")
df_clean.to_csv(CHEMIN_CLEAN, index=False, encoding="utf-8")

print(f"Points de charge : {len(df_clean):,} | Stations : {df_clean['id_station'].nunique():,}"
      f" | Communes : {df_clean['code_insee'].nunique():,}")
print(df_clean.dtypes)


[1_entree] 166,972 -> 166,972 lignes | concernées : 0 | points issus du nettoyage v2
[2_fiable_coherent] 166,972 -> 166,972 lignes | concernées : 113,399 | codes fiables : contrôle par la position
[2_fiable_code_corrige_geo] 166,972 -> 166,972 lignes | concernées : 5,508 | codes fiables : contrôle par la position
[2_fiable_limite_communale] 166,972 -> 166,972 lignes | concernées : 1,817 | codes fiables : contrôle par la position
[2_fiable_coord_suspectes] 166,972 -> 166,972 lignes | concernées : 2,114 | codes fiables : contrôle par la position
[2_fiable_indetermine] 166,972 -> 166,972 lignes | concernées : 258 | codes fiables : contrôle par la position
[2_fiable_non_controle] 166,972 -> 166,972 lignes | concernées : 51 | codes fiables : contrôle par la position
[3_geolocalise] 166,972 -> 166,972 lignes | concernées : 43,636 | code absent ou non fiable remplacé par la commune des coordonnées
[3_sans_commune] 166,972 -> 166,972 lignes | concernées : 189 | code absent ou non fiable, aucun

## Table communale : population, équipement et déserts de recharge

Objectif : une ligne par commune, **y compris les communes sans borne**, pour mesurer les déserts de recharge. Sortie : `data/processed/communes_population_clean.csv`, aucune valeur manquante.

Source de la population : jeu « État de la Base Adresse Nationale par commune » (adresse.data.gouv.fr, Licence Ouverte 2.0), champ `population` = population légale de la commune (population municipale) ; millésime non précisé par le producteur.

| Étape | Règle | Justification |
|---|---|---|
| Arrondissements | Les 45 arrondissements sont regroupés en Paris (75056), Marseille (13055) et Lyon (69123), population additionnée | Le fichier population ne contient que les arrondissements ; les points sont rattachés à la commune |
| Communes retenues | Communes du référentiel 2025 dont la population est connue et non nulle | Les collectivités d'outre-mer absentes du référentiel n'ont pas de contour ; une population nulle (villages détruits de la Meuse) rend tout ratio impossible |
| Périmètre | France métropolitaine et 5 départements d'outre-mer (Guadeloupe, Martinique, Guyane, La Réunion, Mayotte) | Les collectivités d'outre-mer (Saint-Pierre-et-Miquelon, Saint-Barthélemy, Saint-Martin, Wallis-et-Futuna) n'ont aucun point dans IRVE : les garder les compterait comme des déserts sans que l'absence de données soit établie |
| Nom, département, région | Ceux du référentiel 2025 | Identiques au fichier des points ; 17 communes renommées depuis le fichier population |
| Équipement | Nombre de points, de stations et de points de 150 kW ou plus | |
| Distances | Du centre de la commune au point le plus proche, et au point de 150 kW ou plus le plus proche (distance à vol d'oiseau) | |
| Désert de recharge | Commune sans aucun point sur son territoire et à plus de 10 km du point le plus proche | Seuil validé ; sensibilité à 5 et 15 km affichée sous la cellule. La distance part du centre géométrique de la commune : pour les très grandes communes (Arles 757 km², Kourou 2 274 km², Laruns 249 km² ; médiane métropolitaine 11 km²), ce centre est loin des bornes installées en ville ; exiger l'absence de point sur le territoire évite de les classer à tort en désert |
| Désert 150 kW | Commune sans point de 150 kW ou plus sur son territoire et à plus de 30 km d'un tel point | Le règlement (UE) 2023/1804 (AFIR) impose sur le réseau central RTE-T un parc d'au moins 400 kW comprenant un point d'au moins 150 kW tous les 60 km : tout point de l'axe est alors à 30 km au plus d'un parc. Sensibilité à 20 et 50 km affichée |

Colonnes (14) : `code_insee`, `nom_commune`, `code_departement`, `region`, `population`, `latitude`, `longitude` (centre de la commune), `nb_pdc`, `nb_stations`, `nb_pdc_150kw`, `distance_pdc_km`, `distance_150kw_km`, `desert_recharge`, `desert_150kw`. Le nombre de points pour 10 000 habitants se calcule à partir de `nb_pdc` et `population`.


In [14]:
# ===================================================================
# TABLE COMMUNALE : population, équipement et déserts de recharge
# Utilise df_clean, referentiel et COMMUNES_A_ARRONDISSEMENTS (cellules précédentes)
# ===================================================================
CHEMIN_POPULATION = Path("../data/raw/population_communes.csv")
CHEMIN_COMMUNES = Path("../data/processed/communes_population_clean.csv")
SEUIL_DESERT_KM = 10
SEUIL_DESERT_150KW_KM = 30
PUISSANCE_150KW = 150
RAYON_TERRE_KM = 6371.0088
DEPARTEMENTS_OUTRE_MER = ["971", "972", "973", "974", "976"]

journal = []

# --- 1. Population : arrondissements regroupés ---------------------
population = pd.read_csv(CHEMIN_POPULATION, dtype=str, usecols=["code_commune", "population"])
population["population"] = pd.to_numeric(population["population"])
n_pop = len(population)
journaliser("1_population", n_pop, n_pop, 0, "lignes du fichier population")
arrondissement = population["code_commune"].str.fullmatch(
    r"751(0[1-9]|1\d|20)|132(0[1-9]|1[0-6])|6938[1-9]")
population.loc[arrondissement, "code_commune"] = (
    population.loc[arrondissement, "code_commune"].str[:3].map(COMMUNES_A_ARRONDISSEMENTS))
population = population.groupby("code_commune", as_index=False)["population"].sum(min_count=1)
journaliser("2_arrondissements", n_pop, len(population), arrondissement.sum(),
            "arrondissements regroupés en Paris, Marseille et Lyon")

# --- 2. Communes du référentiel 2025 avec population ---------------
hors_referentiel = ~population["code_commune"].isin(referentiel["com_code"])
journaliser("3_hors_referentiel", len(population), len(population) - hors_referentiel.sum(),
            hors_referentiel.sum(),
            "communes absentes du référentiel 2025 (collectivités d'outre-mer) : écartées")
table = referentiel[["com_code", "com_name", "dep_code", "reg_name", "geo_point_2d"]].merge(
    population, left_on="com_code", right_on="code_commune", how="left")
avant = len(table)
table = table.loc[table["population"] > 0].copy()
journaliser("4_population_nulle_ou_absente", avant, len(table), avant - len(table),
            "communes sans population connue ou à population nulle : écartées")
avant = len(table)
table = table.loc[(table["dep_code"].str.len() == 2) | table["dep_code"].isin(DEPARTEMENTS_OUTRE_MER)]
journaliser("4_perimetre", avant, len(table), avant - len(table),
            "collectivités d'outre-mer écartées : périmètre métropole et 5 départements d'outre-mer")
table[["latitude", "longitude"]] = (
    table["geo_point_2d"].str.split(",", expand=True).astype(float).to_numpy())

# --- 3. Équipement par commune -------------------------------------
point_150kw = df_clean["puissance_kw"] >= PUISSANCE_150KW
table["nb_pdc"] = table["com_code"].map(df_clean["code_insee"].value_counts())
table["nb_stations"] = table["com_code"].map(df_clean.groupby("code_insee")["id_station"].nunique())
table["nb_pdc_150kw"] = table["com_code"].map(
    df_clean.loc[point_150kw, "code_insee"].value_counts())
table[["nb_pdc", "nb_stations", "nb_pdc_150kw"]] = (
    table[["nb_pdc", "nb_stations", "nb_pdc_150kw"]].fillna(0).astype(int))
hors_table = ~df_clean["code_insee"].isin(table["com_code"])
journaliser("5_points_hors_table", len(df_clean), len(df_clean), hors_table.sum(),
            "points situés dans une commune écartée (population nulle)")
journaliser("5_communes_sans_point", len(table), len(table), (table["nb_pdc"] == 0).sum(),
            "communes sans aucun point de charge")


# --- 4. Distance au point le plus proche et déserts ----------------
def distance_plus_proche_km(points):
  arbre = BallTree(np.radians(points[["latitude", "longitude"]].to_numpy()), metric="haversine")
  distance, _ = arbre.query(np.radians(table[["latitude", "longitude"]].to_numpy()), k=1)
  return (distance[:, 0] * RAYON_TERRE_KM).round(2)


table["distance_pdc_km"] = distance_plus_proche_km(df_clean)
table["distance_150kw_km"] = distance_plus_proche_km(df_clean.loc[point_150kw])
# Une commune qui a des points sur son territoire n'est pas un désert, même si son centre
# géométrique est éloigné (très grandes communes : Arles, Kourou, Laruns)
loin = table["distance_pdc_km"] > SEUIL_DESERT_KM
loin_150kw = table["distance_150kw_km"] > SEUIL_DESERT_150KW_KM
table["desert_recharge"] = loin & (table["nb_pdc"] == 0)
table["desert_150kw"] = loin_150kw & (table["nb_pdc_150kw"] == 0)
journaliser("6_desert_recharge", len(table), len(table), table["desert_recharge"].sum(),
            f"communes sans point et à plus de {SEUIL_DESERT_KM} km du point le plus proche")
journaliser("6_desert_recharge_ecartes", len(table), len(table), (loin & ~table["desert_recharge"]).sum(),
            "communes éloignées par leur centre mais équipées sur leur territoire : non désertes")
journaliser("6_desert_150kw", len(table), len(table), table["desert_150kw"].sum(),
            f"communes sans point de 150 kW et à plus de {SEUIL_DESERT_150KW_KM} km d'un tel point")

# --- 5. Table finale -----------------------------------------------
df_communes = table.rename(columns={
    "com_code": "code_insee", "com_name": "nom_commune",
    "dep_code": "code_departement", "reg_name": "region"})[[
    "code_insee", "nom_commune", "code_departement", "region", "population",
    "latitude", "longitude", "nb_pdc", "nb_stations", "nb_pdc_150kw",
    "distance_pdc_km", "distance_150kw_km", "desert_recharge", "desert_150kw",
]].reset_index(drop=True)
df_communes["population"] = df_communes["population"].astype(int)
manquants = df_communes.isna().sum()
if manquants.any():
  raise ValueError(f"Valeurs manquantes restantes : {manquants[manquants > 0].to_dict()}")
journaliser("7_sortie", n_pop, len(df_communes), 0,
            f"table communale : {df_communes.shape[1]} colonnes, aucune valeur manquante")
df_communes.to_csv(CHEMIN_COMMUNES, index=False, encoding="utf-8")

# --- 6. Sensibilité des seuils -------------------------------------
habitants = df_communes["population"].sum()
sensibilite = []
for colonne, nombre, seuils in [("distance_pdc_km", "nb_pdc", [5, 10, 15]),
                                ("distance_150kw_km", "nb_pdc_150kw", [20, 30, 50])]:
  for seuil in seuils:
    desert = (df_communes[colonne] > seuil) & (df_communes[nombre] == 0)
    sensibilite.append({
        "indicateur": colonne, "seuil_km": seuil, "communes": desert.sum(),
        "habitants": df_communes.loc[desert, "population"].sum(),
        "part_population_%": round(df_communes.loc[desert, "population"].sum() / habitants * 100, 2)})
print(pd.DataFrame(sensibilite).to_string(index=False))
print(f"\nCommunes : {len(df_communes):,} | Habitants : {habitants:,}")


[1_population] 35,011 -> 35,011 lignes | concernées : 0 | lignes du fichier population
[2_arrondissements] 35,011 -> 34,969 lignes | concernées : 45 | arrondissements regroupés en Paris, Marseille et Lyon
[3_hors_referentiel] 34,969 -> 34,888 lignes | concernées : 81 | communes absentes du référentiel 2025 (collectivités d'outre-mer) : écartées
[4_population_nulle_ou_absente] 34,888 -> 34,875 lignes | concernées : 13 | communes sans population connue ou à population nulle : écartées
[5_points_hors_table] 166,430 -> 166,430 lignes | concernées : 6 | points situés dans une commune écartée (population nulle)
[5_communes_sans_point] 34,875 -> 34,875 lignes | concernées : 24,045 | communes sans aucun point de charge
[6_desert_recharge] 34,875 -> 34,875 lignes | concernées : 695 | communes à plus de 10 km du point le plus proche
[6_desert_150kw] 34,875 -> 34,875 lignes | concernées : 313 | communes à plus de 30 km d'un point de 150 kW ou plus
[7_sortie] 35,011 -> 34,875 lignes | concernées :

Note d'Analyse : Justification de l'enrichissement du dataset IRVE par la Population
1. Contexte et Problématique
Le jeu de données brut des Infrastructures de Recharge pour Véhicules Électriques (IRVE) fournit une vision granulaire de l'offre de bornes de recharge sur le territoire (caractéristiques techniques, géolocalisation, opérateurs, puissance). Cependant, une analyse purement "offre" présente des limites majeures : elle ne permet pas d'évaluer l'adéquation entre le déploiement des infrastructures et les besoins réels des bassins de population.

Pour passer d'une simple description logistique à une véritable analyse territoriale (mobilité durable, déserts de recharge, sur/sous-densité d'équipements), il est indispensable de croiser ces données d'infrastructure avec des données démographiques.

2. Justification de l'ajout du dataset Population
L'intégration du dataset démographique (35 011 lignes, 34 875 communes retenues) répond à trois objectifs analytiques :

Rendre visibles les communes sans borne : la table communale part de toutes les communes, et non des points de recharge ; 24 045 communes sans aucun point y figurent, ce qui permet de mesurer les déserts de recharge.
Pondérer les déserts par la population : 695 communes sont à plus de 10 km du point de recharge le plus proche ; elles regroupent 752 277 habitants (1,10 % de la population).
Normaliser les indicateurs : le nombre de points pour 10 000 habitants permet de comparer des territoires de tailles différentes, à l'échelle du département pour éviter les petits dénominateurs.

3. Méthodologie et Résultat de la Jointure
A. Nettoyage préalable des sources
Dataset IRVE : 226 963 lignes publiées par plusieurs producteurs, ramenées à 166 430 points de charge (une ligne par point, 20 colonnes) ; commune de chaque point contrôlée par sa position à partir des contours communaux 2025 (fichier irve_points_clean.csv).
Dataset Population : jeu « État de la Base Adresse Nationale par commune » (adresse.data.gouv.fr) ; les 45 arrondissements de Paris, Marseille et Lyon sont regroupés en communes ; les communes absentes du référentiel 2025 (collectivités d'outre-mer) ou sans population sont écartées.
B. Réalisation de la jointure
Clé de jointure : code INSEE de la commune (code_insee côté IRVE, code_commune côté population).
Bilan : 166 424 points sur 166 430 sont rattachés à une commune de la table ; les 6 autres sont à Beaumont-en-Verdunois, village détruit de la Meuse à population nulle. La table communale (communes_population_clean.csv) compte 34 875 communes et 14 colonnes, sans valeur manquante.
4. Conclusion
L'enrichissement par la population est une étape structurante indispensable. Il garantit que l'analyse exploratoire et les modélisations prédictives ou géospatiales à venir ne se baseront pas uniquement sur un volume brut d'infrastructures, mais bien sur une mesure d'impact territorial et socio-démographique. 